In [15]:
import os
import torch
import numpy as np
import time
import re
!pip install facenet_pytorch
from facenet_pytorch import InceptionResnetV1, fixed_image_standardization
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [22]:
# --- CONFIGURAZIONE ---
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE_GALLERY = 32  # Per la gallery usiamo i batch (più veloce creare il DB)

# 1. Percorsi
BASE_DIR = './drive/MyDrive/BBA_Dataset_processed'
GALLERY_DIR = os.path.join(BASE_DIR, 'galleries')
PROBES_DIR = os.path.join(BASE_DIR, 'probes')
FEATURES_SAVE_DIR = './drive/MyDrive/BBA_Dataset_processed/Features'
MY_WEIGHTS_PATH = './drive/MyDrive/best_fext.pth'


In [23]:
# --- DATASET HELPER (Solo per la creazione della Gallery) ---
class GalleryDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        # Scansiona cartelle
        subjects = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])

        for subject in subjects:
            # Estrae label numerica dal nome (es. subject_05 -> 5)
            match = re.search(r'\d+', subject)
            label = int(match.group()) if match else -1

            subj_path = os.path.join(root_dir, subject)
            for file in os.listdir(subj_path):
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(subj_path, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

class FaceRecognitionSystem:
    # Aggiungi l'argomento weights_path
    def __init__(self, device, weights_path=None):
        self.device = device
        print(f"Inizializzazione Modello su {self.device}...")

        # 1. Istanzia l'architettura VUOTA (pretrained=None)
        # classify=False è fondamentale: tu vuoi le features (vettore 512),
        # non la classificazione finale su N classi.
        self.resnet = InceptionResnetV1(pretrained=None, classify=False).to(self.device)

        # 2. Carica i tuoi pesi custom
        if weights_path and os.path.exists(weights_path):
            print(f"--- Caricamento pesi custom da: {weights_path} ---")

            # Carica il file .pth (gestisce anche CPU/GPU automaticamente con map_location)
            checkpoint = torch.load(weights_path, map_location=self.device)

            # 3. Estrai solo i pesi del modello dal tuo dizionario
            # Il tuo CheckpointManager salva tutto dentro 'model_state_dict', quindi prendiamo quello.
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
            else:
                state_dict = checkpoint  # Caso fallback se il file fosse solo pesi grezzi

            # 4. Inietta i pesi nel modello
            # strict=False è CRUCIALE qui.
            # Perché? Probabilmente hai addestrato il modello per CLASSIFICARE (aveva un layer finale 'logits'),
            # ma ora lo usi per ESTRARRE FEATURES (senza layer logits).
            # strict=False ignora i pesi del layer 'logits' che non servono più.
            self.resnet.load_state_dict(state_dict, strict=False)

            print(">>> Pesi caricati con successo!")
        else:
            # Fallback: se non dai un percorso, usa quelli standard
            print("ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.")
            self.resnet = InceptionResnetV1(pretrained='vggface2', classify=False).to(self.device)

        # Metti in eval come prima
        self.resnet.eval()

        # ... (resto del codice uguale: gallery_embeddings, transforms, ecc.) ...
        self.gallery_embeddings = None
        self.gallery_labels = None

        # Ricordati le trasformazioni CORRETTE (con Normalize, non fixed_image_standardization)
        self.single_transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    # ------------------------------------------------------------------
    # PARTE 1: COSTRUZIONE GALLERY (Batch Processing -> File .npy)
    # ------------------------------------------------------------------
    def build_and_save_gallery(self, gallery_dir, save_path_prefix):
        print(f"\n--- Costruzione Gallery da: {gallery_dir} ---")
        dataset = GalleryDataset(gallery_dir)

        if len(dataset) == 0:
            print("Nessuna immagine trovata nella gallery!")
            return

        loader = DataLoader(dataset, batch_size=BATCH_SIZE_GALLERY, shuffle=False, num_workers=2)

        emb_list = []
        lbl_list = []

        with torch.no_grad():
            for imgs, lbls in loader:
                imgs = imgs.to(self.device)
                embeddings = self.resnet(imgs)
                embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
                emb_list.append(embeddings.cpu().numpy())
                lbl_list.append(lbls.numpy())

        # Concatena e salva
        self.gallery_embeddings = np.concatenate(emb_list)
        self.gallery_labels = np.concatenate(lbl_list)

        np.save(f"{save_path_prefix}_embeddings.npy", self.gallery_embeddings)
        np.save(f"{save_path_prefix}_labels.npy", self.gallery_labels)

        print(f"Gallery Salvata: {self.gallery_embeddings.shape[0]} volti.")
        print(f"Files: {save_path_prefix}_embeddings.npy, {save_path_prefix}_labels.npy")

        # Carichiamo subito i tensori in GPU per essere pronti all'inferenza
        self._load_gallery_to_gpu()

    def load_existing_gallery(self, load_path_prefix):
        """Carica una gallery precedentemente salvata."""
        print(f"\nCaricamento Gallery da file: {load_path_prefix}...")
        try:
            self.gallery_embeddings = np.load(f"{load_path_prefix}_embeddings.npy")
            self.gallery_labels = np.load(f"{load_path_prefix}_labels.npy")
            self._load_gallery_to_gpu()
            print(f"Gallery caricata: {len(self.gallery_labels)} soggetti.")
        except FileNotFoundError:
            print("Errore: File gallery non trovati.")

    def _load_gallery_to_gpu(self):
        """Metodo interno per spostare la gallery su GPU una volta sola."""
        if self.gallery_embeddings is not None:
            self.gallery_embeddings_tensor = torch.tensor(self.gallery_embeddings).to(self.device)

    # ------------------------------------------------------------------
    # PARTE 2: INFERENZA SINGOLA (One-Shot -> Real Time Simulation)
    # ------------------------------------------------------------------
    def predict_single(self, img_path):
        """
        Prende UN path, estrae features, confronta con tutta la gallery.
        """
        if self.gallery_embeddings is None:
            raise ValueError("La Gallery non è stata caricata! Esegui build o load prima.")

        # 1. Preprocessing Immagine Singola
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            return None, float('inf') # Gestione errore file

        img_tensor = self.single_transform(img)
        img_tensor = img_tensor.unsqueeze(0).to(self.device) # Aggiunge dimensione batch (1, 3, 160, 160)

        # 2. Estrazione Feature (Forward Pass)
        with torch.no_grad():
            probe_emb = self.resnet(img_tensor) # (1, 512)
            probe_emb = torch.nn.functional.normalize(probe_emb, p=2, dim=1)

        # 3. Calcolo Distanza Euclidea (Probe vs Tutta la Gallery)
        # torch.cdist calcola la distanza tra il vettore probe e TUTTI i vettori gallery in parallelo
        dists = torch.cdist(probe_emb, self.gallery_embeddings_tensor, p=2)

        # 4. Trova il minimo
        min_dist, min_idx = torch.min(dists, dim=1)

        match_idx = min_idx.item()
        distance = min_dist.item()

        predicted_label = self.gallery_labels[match_idx]

        return predicted_label, distance

    def print_memory(self):
        if self.device.type == 'cuda':
            alloc = torch.cuda.memory_allocated(self.device) / 1024**2
            print(f"[Memory Check] GPU Allocata: {alloc:.2f} MB")



In [26]:

# 2. Inizializza Sistema
system = FaceRecognitionSystem(DEVICE)

# ---------------------------------------------------------
# STEP A: CREAZIONE GALLERY (Eseguito una volta sola)
# ---------------------------------------------------------
# Questo metodo legge tutte le cartelle gallery, crea i batch, e salva i file .npy
system.build_and_save_gallery(GALLERY_DIR, FEATURES_SAVE_DIR)

# (Opzionale) Se hai già i file .npy puoi commentare la riga sopra e usare:
# system.load_existing_gallery(FEATURES_SAVE_DIR)

# ---------------------------------------------------------
# STEP B: INFERENZA SINGOLA SU PROBES
# ---------------------------------------------------------
print("\n--- Inizio Test Inferenza Singola (Simulazione Real-Time) ---")

# Raccogliamo i file probes manualmente per simulare un loop
probe_files = []
for root, _, files in os.walk(PROBES_DIR):
    for file in files:
        if file.lower().endswith(('.jpg', '.png', '.jpeg')):
            # Salviamo anche la vera label per verificare se ci ha preso (opzionale)
            # Assumendo che il parent folder sia "subject_xx"
            true_label_match = re.search(r'\d+', os.path.basename(root))
            true_label = int(true_label_match.group()) if true_label_match else -1

            probe_files.append((os.path.join(root, file), true_label))

# Ordiniamo per pulizia
probe_files.sort()

correct_count = 0
total_count = 0

# Limitiamo a 50 immagini per testare, rimuovi [:50] per farle tutte
for img_path, true_label in probe_files:

    start_time = time.time()

    # === CHIAMATA CRUCIALE: INFERENZA SINGOLA ===
    pred_label, distance = system.predict_single(img_path)
    # ============================================

    elapsed = time.time() - start_time

    # Verifica risultato
    is_correct = (pred_label == true_label)
    if is_correct: correct_count += 1
    total_count += 1

    print(f"Probe: {os.path.basename(img_path)} | Pred: {pred_label} (Vero: {true_label}) | Dist: {distance:.4f} | Time: {elapsed:.3f}s")

    # Controllo memoria ogni 10 iterazioni per vedere se "scoppia"
    if total_count % 10 == 0:
        system.print_memory()

# Risultati finali
if total_count > 0:
    acc = (correct_count / total_count) * 100
    print(f"\n--- FINE TEST ---")
    print(f"Accuratezza Totale: {acc:.2f}% ({correct_count}/{total_count})")

Inizializzazione Modello su cuda:0...
ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.

--- Costruzione Gallery da: ./drive/MyDrive/BBA_Dataset_processed/galleries ---
Gallery Salvata: 152 volti.
Files: ./drive/MyDrive/BBA_Dataset_processed/Features_embeddings.npy, ./drive/MyDrive/BBA_Dataset_processed/Features_labels.npy

--- Inizio Test Inferenza Singola (Simulazione Real-Time) ---
Probe: subj_00_left_side_light_00.jpg | Pred: 0 (Vero: 0) | Dist: 0.6617 | Time: 0.022s
Probe: subj_00_left_side_light_01.jpg | Pred: 0 (Vero: 0) | Dist: 0.5949 | Time: 0.021s
Probe: subj_00_left_side_light_02.jpg | Pred: 0 (Vero: 0) | Dist: 0.6445 | Time: 0.020s
Probe: subj_00_light_behind_subj_00.jpg | Pred: 0 (Vero: 0) | Dist: 0.6437 | Time: 0.020s
Probe: subj_00_light_behind_subj_01.jpg | Pred: 0 (Vero: 0) | Dist: 0.6423 | Time: 0.020s
Probe: subj_00_light_behind_subj_02.jpg | Pred: 0 (Vero: 0) | Dist: 0.6552 | Time: 0.020s
Probe: subj_00_low_light_00.jpg | Pred: 0 (Vero: 0) | Dis